# Document RAG QA (LangChain, FAISS & HuggingFace / OpenAI)
Build a local **FAISS** vector store and query grounded facts using **Llama 3.3 (HuggingFace)** and **Sentence-Transformers Embeddings**.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
openai_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN")

if openai_key and not openai_key.startswith("your_"):
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
    print("OpenAI Embeddings & LLM initialized successfully.")
elif hf_token and not hf_token.startswith("your_"):
    from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFaceEndpointEmbeddings
    embeddings = HuggingFaceEndpointEmbeddings(
        model="sentence-transformers/all-MiniLM-L6-v2",
        huggingfacehub_api_token=hf_token
    )
    endpoint = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-3.3-70B-Instruct",
        task="conversational",
        huggingfacehub_api_token=hf_token
    )
    llm = ChatHuggingFace(llm=endpoint)
    print("HuggingFace Embeddings & LLM initialized successfully.")
else:
    raise ValueError("Please configure HUGGINGFACEHUB_API_TOKEN or OPENAI_API_KEY in .env")


HuggingFace Embeddings & LLM initialized successfully.


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Sample Knowledge Base Documents
sample_documents = [
    "FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search and clustering of dense vectors.",
    "LangChain is a framework designed to simplify the creation of applications using large language models (LLMs).",
    "LangGraph is a library for building stateful, multi-actor applications with LLMs using graph-based state machines.",
    "Retrieval-Augmented Generation (RAG) combines external knowledge retrieval with generative LLMs to produce factually grounded answers."
]

# Create FAISS Vector Index
vectorstore = FAISS.from_texts(sample_documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("FAISS vector store indexed successfully! Ready for queries.")


FAISS vector store indexed successfully! Ready for queries.


In [4]:
# RAG Query Function
rag_prompt = PromptTemplate.from_template(
"""You are a helpful assistant. Answer the question strictly using the provided context.

Context:
{context}

Question: {question}

Answer:"""
)

def ask(question: str):
    matched_docs = retriever.invoke(question)
    context_str = "\n\n".join(doc.page_content for doc in matched_docs)
    formatted_prompt = rag_prompt.format(context=context_str, question=question)
    res = llm.invoke(formatted_prompt)
    
    print(f"\n❓ QUESTION: {question}")
    print("="*50)
    print("💡 ANSWER:")
    print(res.content if hasattr(res, 'content') else str(res))

# Execute RAG test query
ask("Who is the pm of india?")



❓ QUESTION: Who is the pm of india?
💡 ANSWER:
There is no information in the provided context about the Prime Minister of India. The context only mentions LangGraph and LangChain, which are related to building applications with large language models, and does not provide any information about political figures or governments.
